This notebook aim to save and combine data akademik dan data pedoman akademik

### Data Pedoman Akademik (Dense Method)

In [45]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
        "https://akademik.nurulfikri.ac.id/2-administrasi/"
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
        "administrasi MBKM"
    ],
}

In [134]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [39]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [133]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

In [135]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [136]:
embeddings = IndoBertEmbeddings()

In [137]:
from langchain_elasticsearch import ElasticsearchStore

In [138]:
from langchain_elasticsearch import DenseVectorStrategy


vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=DenseVectorStrategy(hybrid=True)
)

In [46]:
vector_store.add_documents(docs_splits)

2026-02-08 10:32:52,406 - INFO - HEAD http://localhost:9200/langchain_index [status:404 duration:0.004s]
2026-02-08 10:32:52,574 - INFO - PUT http://localhost:9200/langchain_index [status:200 duration:0.098s]
2026-02-08 10:32:58,166 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.256s]


['abdd8263-08a0-4601-8582-2c4021e50b0f',
 '4884d879-885e-4a6c-9205-d4b4c71ccaad',
 'e657b347-71ae-48ae-a4c7-d8655ee3a424',
 'e0765d45-70df-411e-8b22-9891ca453961',
 'd2c55c45-4364-41e2-bc80-82c26547ddbc',
 '7f317b18-bb94-4af7-86ed-45a5d7d6b707',
 'e7da3354-e82b-4f7b-9711-8a9af54ddd65',
 'e79575ed-6210-4056-93c8-5a9d46dcd6e3',
 '242edeec-861a-4396-a6a9-d517ccfe7606',
 'b4bbdefa-f46b-48bb-a221-e23342a0e761',
 'cf633ca5-1662-4d46-bdbb-fec2a5a3a0ee',
 '789803fb-5b75-4357-8802-f7a7ffde8e7e',
 '38de5c9d-8a6d-430d-98f5-e90375980552',
 'd8d5a929-0576-4930-8a4c-3f781f29d579',
 'd6edeb5d-0916-4300-a938-8922a06e3b1c',
 '0dc5a248-38dc-4c92-b709-357f40b387b3',
 '34e5b2d3-d2d2-4bf8-8c33-478311a7966f',
 '9abecbb7-b828-40bd-9213-73c9fb956e65',
 'a26522aa-3511-4ee5-81d1-bca0fa3f047a',
 '41568d81-46b7-413f-a170-f35696797331',
 '85237228-3c39-4412-b104-14015866bb4f',
 '2e185ece-0e4f-4dfe-bf5d-2c8edb5084d5',
 '21a2e366-4c0e-4e95-b447-2224cd93d765',
 'c5a642d9-867e-43bf-b498-c3b978177b97',
 'eb4d7d3e-832b-

### Testing Dense Retriever

In [139]:
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={
        "k":5
    }
)


In [140]:
retriever.invoke("Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri", k=3)

[Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/', 'title': '3. Kode Etik Mahasiswa – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='Pasal 4\nManfaat Kode Etik Mahasiswa Sekolah Tinggi Teknologi Terpadu Nurul Fikri adalah:\n\nMemberikan panduan untuk mahasiswa dalam beretika dan berperilaku.\nMemberikan koreksi diri untuk mahasiswa dalam hal beretika dengan baik.\nMemberi aturan dalam pergaulan antar mahasiswa dan antara mahasiswa dengan sivitas akademika lain di lingkungan Sekolah Tinggi Teknologi Terpadu Nurul Fikri.\nMemelihara suasana, lingkungan serta fasilitas atau sarana prasarana di Sekolah Tinggi Teknologi Terpadu Nurul Fikri.\n\nBAB III\nHAK DAN KEWAJIBAN MAHASISWA\nPasal 5\nMahasiswa Sekolah Tinggi Teknologi Terpadu Nurul Fikri mempunyai hak antara lain:'),
 Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/', 'title': '1. Syarat Kelulusan – Pedoman Akademik STT-NF', 'language': 'en-US'}, 

### Data Akademik Mahasiswa (Sparse Method)

In [48]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa-2.xlsx'
]

### Processing document

In [49]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [50]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [51]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [55]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-02-15 17:15:45,823 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2026-02-15 17:15:45,853 - INFO - Going to convert document batch...
2026-02-15 17:15:45,854 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-02-15 17:15:46,221 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-02-15 17:15:46,221 - INFO - Loading plugin 'docling_defaults'
2026-02-15 17:15:46,224 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-15 17:15:46,225 - INFO - Processing document sintetik-data-akademik-mahasiswa-2.xlsx
2026-02-15 17:15:46,226 - INFO - Processing sheet 0: sintetik-data-akademik
2026-02-15 17:15:46,494 - INFO - Finished converting document sintetik-data-akademik-mahasiswa-2.xlsx in 0.68 sec.


In [141]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="test_index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

In [57]:
vector_store_sparse.add_documents(data_akademik_split)

2026-02-15 17:15:51,893 - INFO - HEAD http://localhost:9200/test_index [status:404 duration:0.004s]
2026-02-15 17:15:52,035 - INFO - PUT http://localhost:9200/test_index [status:200 duration:0.141s]
2026-02-15 17:15:52,098 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.062s]


['69adf126-b36b-4fe1-af6f-f3e20fd9c424',
 '95255f2e-0895-4abf-84b6-b8ae82ddcb8b',
 'e959e169-712f-48fb-9d41-8b28a965aec0',
 '4c4f21bf-9161-4804-bded-44d92ca96881',
 'c561d2cc-82f6-4342-96db-9fb1484da865',
 '3e6af201-3cf0-462c-90f5-ebacdf4be888',
 'e27911ad-2605-4add-97a5-e497a37f7f8c',
 '229aac3d-c695-44dc-9db3-5a6af3ab95ae',
 '37af4eec-0b8a-44e7-ba7f-f9702d46b2e7',
 '7b109f71-d98e-4c7f-8fbb-e027da05fcaa',
 '8e404df7-4350-4890-96fe-62d6e5207ee7',
 '803f470e-0c50-4a73-b56a-2aa574de19a1',
 '4ed1adf0-f871-4937-a867-279a6e5bc4d4',
 'b23721ef-98df-4942-8efd-c96b15ed1c67',
 'ea346655-d7d7-4e4b-8b1b-4377caf597aa',
 '51b3fd7b-b109-46b3-b7fc-fe04a962d1cf',
 'add5ac5f-320b-4dd6-86bf-60b039bb89af',
 '0ecef333-328f-41ee-8238-7eb32a44de65',
 'df95b4f1-82c2-4f99-8637-f9242ade63c7',
 'b964273b-9b75-4479-80c8-ebfebd66fa07',
 '71379385-908b-4416-8dbf-68bf33348960',
 '411e11fa-63ea-455e-80de-3a9503bdfbfe',
 'fa984bb2-948a-4da9-ae4d-993708c6eb72',
 '80bc0a1c-6ab3-439b-b9da-5edfa3a2a0a2',
 'b4fb1015-b0bb-

In [10]:
vector_store_sparse.similarity_search("berapa ipk romi wahyudi")

[Document(metadata={}, page_content='|   No |         NIM | Nama Mahasiswa      | Jurusan               |   Semester |   Total SKS |   IPS |   IPK |   Kehadiran (%) | Status   | Dosen Pembimbing              |\n|------|-------------|---------------------|-----------------------|------------|-------------|-------|-------|-----------------|----------|-------------------------------|\n|    1 | 2.02401e+09 | Tono Setiawan       | Teknik Informatika    |          1 |          20 |  3.4  |  3.37 |              95 | Aktif    | Ir. Sigit Santoso, M.Kom      |\n|    2 | 2.02401e+09 | Tri Pratama         | Teknik Informatika    |          2 |          45 |  3.81 |  3.76 |              81 | Aktif    | Ir. Lestari Handayani, M.Kom  |\n|    3 | 2.02003e+09 | Agus Nasution       | Bisnis Digital        |         10 |         148 |  3.83 |  3.98 |              78 | Lulus    | Dr. Rina Novita, M.Kom        |'),
 Document(metadata={}, page_content='|  123 | 2.02304e+09 | Iman Basuki         | Manajemen

### Create Agent Which can separate the work to do the conditioning

In [142]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import operator
from langchain_core.messages import AIMessage

### Tools 

In [145]:

@tool
def query_from_academic_rule(query: str):
    """
    Gunakan tool ini untuk query dari user yang HANYA bermaksud untuk pertanyaan seputar ATURAN, KEBIJAKAN, SYARAT, atau PROSEDUR KAMPUS.
    
    args: 
        query: Search terms to look for
    """
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data pedoman akademik."
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari Pedoman Akademik:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi Kesalahan saat mengakses vector database"


@tool
def get_student_academic_record(query: str):
    """
    Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.
    Gunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.
    
    Args:
        query: Search terms to look for
    """
    try:
        docs = vector_store_sparse.similarity_search(query)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data akademik mahasiswa"
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari data akademik mahasiswa:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi kesalahn saat mengakses data akademik"

In [283]:
import json
import re
import ast
from typing import Annotated, List, TypedDict, Optional

from langchain_core.messages import (
    AIMessage, 
    ToolMessage, 
    SystemMessage, 
    HumanMessage, 
    AnyMessage
)
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
        
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        
        # Conditional edge sekarang lebih pintar
        graph.add_conditional_edges( 
            "llm", self.exists_action, {True: "action", False: END} 
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        
        self.graph = graph.compile()

    # --- FUNGSI HELPER (Jantung Perbaikan) ---
    def _try_extract_tool_calls(self, content: str) -> List[dict]:
        """
        Mencoba mengekstrak dan memvalidasi JSON tool calls dari string.
        Mengembalikan list kosong [] jika gagal parsing.
        """
        if not content:
            return []
            
        try:
            matches = re.finditer(r'(\{.*?\}|\[.*?\])', content, re.DOTALL)
            
            candidates = []
            for match in matches:
                str_val = match.group()
                if '"name"' in str_val or "'name'" in str_val:
                    candidates.append(str_val)
            
            if not candidates:
                start = next((i for i, c in enumerate(content) if c in '{['), -1)
                end = next((i for i in range(len(content)-1, -1, -1) if content[i] in '}]'), -1)
                if start != -1 and end != -1 and start <= end:
                    candidates.append(content[start : end+1])

            extracted_tools = []
            for candidate in candidates:
                try:
                    # Prioritas 1: JSON Standard
                    data = json.loads(candidate)
                except:
                    try:
                        # Prioritas 2: Python Literal (untuk single quotes)
                        data = ast.literal_eval(candidate)
                    except:
                        continue # Skip jika gagal total

                # Normalisasi ke List
                if isinstance(data, dict):
                    data = [data]
                
                # Validasi struktur: Harus punya 'name'
                for item in data:
                    if isinstance(item, dict) and ('name' in item or 'tool' in item):
                        # Bersihkan args
                        raw_args = item.get('args') or item.get('arguments') or item.get('parameters') or {}
                        if isinstance(raw_args, str):
                            try:
                                raw_args = json.loads(raw_args)
                            except:
                                pass
                        
                        extracted_tools.append({
                            'name': item.get('name') or item.get('tool'),
                            'args': raw_args,
                            'id': f"manual_{len(extracted_tools)}"
                        })
            
            return extracted_tools

        except Exception:
            return []

    # --- EXISTS ACTION (Penjaga Pintu Yang Ketat) ---
    def exists_action(self, state: AgentState) -> bool:
        result = state['messages'][-1]
        
        # 1. Cek Native (Selalu prioritas)
        if hasattr(result, "tool_calls") and len(result.tool_calls) > 0:
            return True
            
        # 2. Cek Manual: Hanya return True jika KITA BISA MEM-PARSINGNYA
        content = result.content if result.content else ""
        extracted = self._try_extract_tool_calls(content)
        
        if extracted:
            print(f"🕵️ Valid JSON Tool Call ditemukan: {len(extracted)} tools.")
            return True
        
        return False

    def call_model(self, state: AgentState):
        # (Kode call_model sama seperti sebelumnya)
        messages = state["messages"]
        last_message = messages[-1]
        if isinstance(last_message, HumanMessage):
             if "(Remember: If you need to act" not in last_message.content:
                new_content = last_message.content + "\n(Remember: If you need to act, output ONLY JSON tool call, do not chat.)"
                messages = messages[:-1] + [HumanMessage(content=new_content)]
        
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
            
        try:
            response = self.model.invoke(messages)
            return {'messages': [response]}
        except:
            return {'messages': [AIMessage(content="System Error.")]}

    # --- TAKE ACTION (Eksekutor) ---
    def take_action(self, state: AgentState):
        llm_message = state['messages'][-1]
        tools_to_run = []
        results = []

        # 1. Ambil tools (pasti berhasil karena sudah dicek di exists_action)
        if hasattr(llm_message, "tool_calls") and len(llm_message.tool_calls) > 0:
            tools_to_run = llm_message.tool_calls
        else:
            # Gunakan helper yang sama persis
            tools_to_run = self._try_extract_tool_calls(llm_message.content)

        # 2. Safety Check Terakhir (Anti-Loop)
        # Kalau entah bagaimana tools_to_run tetap kosong (sangat jarang terjadi dgn logika baru),
        # JANGAN return ToolMessage error. Return pesan kosong atau pesan AI untuk mengakhiri giliran.
        if not tools_to_run:
            print("⚠️ Anomali: Masuk take_action tapi tools kosong. Force End.")
            # Kita return pesan AI biasa untuk "mengaku kalah" dan memutus flow action
            return {"messages": [AIMessage(content="Maaf, saya mencoba menjalankan perintah sistem tapi format data tidak terbaca.")]}

        # 3. Eksekusi
        for tool_call in tools_to_run:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call.get('id')

            if tool_name not in self.tools:
                results.append(ToolMessage(
                    tool_call_id=tool_id, 
                    name=tool_name, 
                    content=f"Error: Tool '{tool_name}' not found."
                ))
                continue

            try:
                # Invoke Tool
                tool_output = self.tools[tool_name].invoke(tool_args)
            except Exception as e:
                tool_output = f"Error executing tool: {str(e)}"

            results.append(ToolMessage(
                tool_call_id=tool_id,
                name=tool_name,
                content=str(tool_output)
            ))

        return {"messages": results}

### LLM model

In [284]:
from langchain_ollama import ChatOllama

In [285]:
system_prompt = """
Kamu adalah Asisten Akademik Kampus yang cerdas. Tugasmu adalah menjawab pertanyaan user.
Kamu memiliki akses ke dua alat:
1. `query_from_academic_rule`: Untuk mencari aturan umum (Pedoman).
2. `get_student_academic_record`: Untuk mencari data pribadi mahasiswa (Database).

STRATEGI ROUTING:
- Jika user bertanya ATURAN UMUM KAMPUS -> Gunakan `query_from_academic_rule`.
- Jika user bertanya DATA PRIBADI MAHASISWA -> Gunakan `get_student_academic_record`.
- Jika user bertanya KEDUANYA (misal: "Apakah saya memenuhi syarat?"), panggil KEDUA alat tersebut.
- Jika user hanya menyapa (Halo/Hi) -> JANGAN panggil alat, jawab langsung dengan sopan.
"""

system_prompt_binary = """
Kamu adalah staf Akademik Kampus yang cerdas. Kamu punya akses terhadap informasi kampus menggunakan dua tools berikut:
1. `query_from_academic_rule`: Untuk mencari aturan atau Pedoman akademik.
2. `get_student_academic_record`: Untuk mencari data pribadi mahasiswa.

STRATEGI ROUTING:
- Jika user bertanya ATURAN UMUM KAMPUS atau PEDOMAN AKADEMIK -> Gunakan `query_from_academic_rule`.
- Jika user bertanya DATA PRIBADI MAHASISWA -> Gunakan `get_student_academic_record`.
- Jika user bertanya KEDUANYA ANTARA PEDOMAN AKADEMIK DAN DATA MAHASISWA, panggil KEDUA alat tersebut.
- Jika user hanya menyapa (Halo/Hi) -> JANGAN panggil alat, jawab langsung dengan sopan.

CONTOH KASUS (FEW-SHOT):
User: "Apakah mahasiswa semester 1 boleh cuti?"
Thought: User tanya aturan cuti semester 1. Tidak ada data Mahasiswa yang diminta.
Action: `query_from_academic_rule`

User: "Apakah Tono Setiawan (2024010001) statusnya aktif?"
Thought: User tanya status orang spesifik (Tono). Tidak butuh aturan umum.
Action: `get_student_academic_record`

User: "Apakah IPK Rizky Kusuma cukup untuk mengambil 24 SKS?"
Thought: Saya butuh data IPK Rizky (Tool Mahasiswa) DAN saya butuh aturan batas SKS berdasarkan IPK (Tool Aturan).
Action: `get_student_academic_record` & `query_from_academic_rule`

User: "Halo, selamat pagi."
Thought: Hanya sapaan.
Action: None (Direct Answer).

Lets think step by step
"""

System_prompt_CoT = """ 
    Kamu adalah staf kampus yang bisa memberikan informasi yang diminta User. 
    Hal pertama yang kamu lakukan adalah BREAK DOWN pertanyaan User.

    Continue
    Jika user Hanya menyapa, jawab langsung dengan sopan

    Continue
    Jika user Butuh Informasi Pedoman akademik, Persyaratan, Aturan, atau Rule
    Panggil tools `query_from_academic_rule`
    Contoh: Apa Syarat lulus di kampus STTNF 

    Continue
    Jika user Butuh informasi Akademik Mahasiswa (Biasanya User akan memberikan Atribut tertentu dari mahasiswa)
    Panggil tools `get_student_academic_record` dengan query Atribut mahasiswa
    Contoh : Berikan informasi dari mahasiswa dengan NIM atau Dengan Nama berikut

    Continue 
    Jika user Butuh informasi dari keduanya antara Pedoman Akademik dan Informasi akademik mahasiswa (dengan atribut tertentu)
    Panggil tools `query_from_academic_rule` dan `get_student_academic_record`
    Contoh: Apakah mahasiswa atas Nama tertentu (atribut) IPK-nya Bisa Untuk lulus (Persyaratan)

    Let's think step by step and answer ini bahasa indonesia

"""




In [286]:
model = ChatOllama(
    model="mistral:7b-instruct-v0.3-q8_0", 
    temperature=0, 
    streaming=True)

In [287]:
tools = [query_from_academic_rule, get_student_academic_record]

In [288]:
tools[1]

StructuredTool(name='get_student_academic_record', description='Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.\nGunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.\n\nArgs:\n    query: Search terms to look for', args_schema=<class 'langchain_core.utils.pydantic.get_student_academic_record'>, func=<function get_student_academic_record at 0x14a12f060>)

In [289]:
bot = Agent(model, tools, system=System_prompt_CoT)

In [ ]:
result = bot.graph.invoke({"messages": [HumanMessage(content="selamat siang")]})

🕵️ Valid JSON Tool Call ditemukan: 1 tools.


In [291]:
print(result['messages'][-1].content)

 Dari data yang diberikan, Mega Anggraini adalah mahasiswa semester 5 dan memiliki jurusan Sistem Informasi dengan NIM 2.02402e+09.

Selamat mencari informasi! Jika ada pertanyaan lain, jangan ragu untuk menanyakan.


### Bagian ini digunakan untuk mengevaluasi sistem RAG, baik untuk proses `Retrieval`, maupun pengujian hasil `Generation` dari model LLM

In [56]:
import json
import numpy as np


def calculate_metric(retrieved_docs, ground_truth_source, k=5):
    top_k_docs = retrieved_docs[:k]

    # ambil sumber datanya (asumsi data pedoman akademik akan punya atribut sumber data)
    retrieved_sources = [doc.metadata.get('source') for doc in top_k_docs]

    # apakah URL yang benar ada di dalam list yang ditemukan
    if ground_truth_source in retrieved_sources:
        hit_score = 1
        recall_score = 1
    else:
        hit_score = 0
        recall_score = 0

    # berapa persen dokumen di Top K yang benar
    relevant_count = retrieved_sources.count(ground_truth_source)
    precision_score = relevant_count / k
    return hit_score, precision_score, recall_score

In [57]:
def evaluate_rag_system(dataset, retrieval_function, k_values=[1, 3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']
        gt_source = data['ground_truth_source']

        # query dengan vector store es
        retrieved_docs = retrieval_function.invoke(query)

        # hitung score
        for k in k_values:
            hit, prec, rec = calculate_metric(retrieved_docs, gt_source, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)

    
    # rata-rata
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall{k}'] = np.mean(results[k]['recall'])

    return final_report

In [58]:
dataset = [
    {
        "question": "Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/"
    },
    {
        "question": "Bisa jelaskan sejarah sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-sejarah/",
    },
    {
        "question": "Bisa berikan informasi mengenai Ahmad Rio Adriansyah",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/4-profil-dosen/"
    },
    {
        "question": "kapan evaluasi akademik dilaksanakan",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/"
    },
    {
        "question": "Apa etika mahasiswa terhadap dosen di sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/"
    },
    {
        "question": "Apa tugas dan kewajiban dosen pembimbing MBKM",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/2-administrasi/"
    },
    {
        "question": "Berikan Alur pengajuan surat untuk magang di luar mitra dikti dan internal kampus",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/2-administrasi/"
    },
    {
        "question": "berikan penjelasan mengenai kebijakan otonomi keilmuan di sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
         
    },
    {
        "question": "Bagaimana penangan untuk mahasiswa yang sudah mempunyai masa studi 6 tahun",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
         
    },
      {
        "question": "Bagaimana penangan untuk mahasiswa yang sudah mempunyai masa studi 6 tahun",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
         
    }
]   

In [66]:
report = evaluate_rag_system(dataset, retriever, [3, 5])

In [67]:
import pprint
pprint.pprint(report)

{'Hit_Rate@3': 0.9,
 'Hit_Rate@5': 0.9,
 'Precision3': 0.6666666666666666,
 'Precision5': 0.62,
 'Recall3': 0.9,
 'Recall5': 0.9}


### Pengujian Data Akademik Mahasiswa

In [88]:
# Ground truth Source

Dataset = [
    {
        "question":"Apa jurusan Kartika Siregar",
        "ground_truth_answer":"sistem informasi"
    },
    {
        "question":"Berapa persen kehadiran Oscar Wulandari",
        "ground_truth_answer":"98"
    },
    {
        "question":"Berapa IPK dari Aditya Anggraini",
        "ground_truth_answer":"3.48"
    },
    {
        "question":"Berapa IPS dari Xavier Purnomo",
        "ground_truth_answer":"3.18"
    },
    {
        "question":"Siapa dosen pembimbing Oscar Kusuma",
        "ground_truth_answer":"H. Miko Maulana, M.Kom"
    },
    {
        "question":"Mahasiswa dengan NIM 2022030032",
        "ground_truth_answer":"Ahmad Utami"
    },
     {
        "question":"status mahasiswa atas nama Iwan Purnomo",
        "ground_truth_answer":"lulus"
    },
    {
        "question":"kehadiran Tri Astuti",
        "ground_truth_answer":"100"
    },
     {
        "question":"jurusan Bayu Siregar",
        "ground_truth_answer":"Teknik informatika"
    },
    {
        "question":"Berapa Total sks Miko Ramadhan",
        "ground_truth_answer":"122"
    },
]


In [89]:
def calculate_metric_sparse(retrieved_docs, ground_truth, k=5):

    # get Top-k documents
    top_k_docs = retrieved_docs[:k]

    relevant_list = []

    for doc in top_k_docs[:k]:
        is_relevant = False

        doc_content = doc.page_content.lower()
        gt_token = str(ground_truth).lower()
        if gt_token in doc_content:
            is_relevant = True

        relevant_list.append(is_relevant)
    
    if any(relevant_list):
        hit_score = 1
        recall_score = 1

    else:
        hit_score = 0
        recall_score = 0

    precision_score = sum(relevant_list) / k
    return hit_score, precision_score, recall_score

In [90]:
import numpy as np

In [91]:
def evaluate_rag_system_sparse(dataset, retrieval_function,    k_values=[3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']

        gt_data = data['ground_truth_answer']
        retrieved_docs = retrieval_function.similarity_search(gt_data)

        for k in k_values:
            hit, prec, rec = calculate_metric_sparse(retrieved_docs, gt_data, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)
        
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision@{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall@{k}'] = np.mean(results[k]['recall'])

    return final_report

In [92]:
report = evaluate_rag_system_sparse(
    Dataset,
    vector_store_sparse,
)

2026-02-15 20:55:07,538 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.010s]
2026-02-15 20:55:07,550 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.011s]
2026-02-15 20:55:07,559 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.008s]
2026-02-15 20:55:07,565 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.006s]
2026-02-15 20:55:07,574 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.008s]
2026-02-15 20:55:07,580 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.005s]
2026-02-15 20:55:07,585 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.005s]
2026-02-15 20:55:07,590 - I

In [93]:
import pprint
pprint.pprint(report)

{'Hit_Rate@3': 1.0,
 'Hit_Rate@5': 1.0,
 'Precision@3': 0.9333333333333332,
 'Precision@5': 0.74,
 'Recall@3': 1.0,
 'Recall@5': 1.0}
